# Data Cleaning

---

## Objective

Clean the Online Retail dataset to ensure high-quality customer-product interactions for recommendation modeling.

---

## Business Perspective

Recommendation systems rely heavily on valid customer purchase behavior.

Including cancelled transactions, invalid purchases, or anonymous customers may produce misleading recommendations and negatively impact business decisions.

---

## Data Science Perspective

Data cleaning ensures that the recommendation model learns only from meaningful interactions.

Removing invalid observations improves model quality, reduces noise, and increases recommendation reliability.

---

## Methodology

The following cleaning procedures will be performed:

1. Remove rows with missing CustomerID.
2. Remove rows with missing Description.
3. Investigate duplicate records.
4. Remove cancellation transactions.
5. Remove negative quantity transactions.
6. Remove invalid prices.
7. Validate cleaned dataset.
8. Save cleaned dataset.

---

In [1]:
# ==================================================
# IMPORT LIBRARIES
# ==================================================

from pathlib import Path
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)

print("Libraries imported successfully.")

Libraries imported successfully.


In [2]:
# ==================================================
# LOAD DATASET
# ==================================================

PROJECT_ROOT = Path().resolve().parent

DATA_PATH = PROJECT_ROOT / "data" / "raw" / "Online Retail.xlsx"

df = pd.read_excel(DATA_PATH)

print("Original Dataset Shape:", df.shape)

Original Dataset Shape: (541909, 8)


In [3]:
# ==================================================
# CREATE WORKING COPY
# ==================================================

df_clean = df.copy()

print("Working copy created.")

Working copy created.


In [4]:
# ==================================================
# MISSING VALUE CHECK
# ==================================================

missing_before = pd.DataFrame({
    "missing_count": df_clean.isnull().sum(),
    "missing_percentage":
    round(df_clean.isnull().mean()*100,2)
})

display(missing_before)

,missing_count,missing_percentage
InvoiceNo,0,0.00
StockCode,0,0.00
Description,1454,0.27
Quantity,0,0.00
InvoiceDate,0,0.00
UnitPrice,0,0.00
CustomerID,135080,24.93
Country,0,0.00


## Cleaning Decision 1

Rows without CustomerID cannot be used in recommendation systems because customer-product interactions cannot be established.

Rows with missing Description also provide limited business value and will be removed.

In [5]:
# ==================================================
# REMOVE MISSING VALUES
# ==================================================

rows_before = len(df_clean)

df_clean = df_clean.dropna(
    subset=["CustomerID", "Description"]
)

rows_after = len(df_clean)

print("Rows Removed :", rows_before - rows_after)
print("Remaining Rows :", rows_after)

Rows Removed : 135080
Remaining Rows : 406829


In [6]:
# ==================================================
# DUPLICATE CHECK AFTER MISSING VALUE REMOVAL
# ==================================================

duplicate_count = df_clean.duplicated().sum()

print("Duplicate Rows:", duplicate_count)

Duplicate Rows: 5225


## Cleaning Decision 2

Duplicate transactions may introduce bias because the same interaction can be counted multiple times.

Exact duplicate records will be removed to avoid overestimating customer-product interactions.

In [7]:
# ==================================================
# REMOVE DUPLICATE ROWS
# ==================================================

rows_before = len(df_clean)

df_clean = df_clean.drop_duplicates()

rows_after = len(df_clean)

print("Duplicate Rows Removed :", rows_before - rows_after)
print("Remaining Rows :", rows_after)

Duplicate Rows Removed : 5225
Remaining Rows : 401604


In [8]:
# ==================================================
# CANCELLATION TRANSACTION CHECK
# ==================================================

cancelled_rows = (
    df_clean["InvoiceNo"]
    .astype(str)
    .str.startswith("C")
    .sum()
)

cancel_percentage = round(
    cancelled_rows / len(df_clean) * 100,
    2
)

print("Cancelled Transactions :", cancelled_rows)
print("Percentage :", cancel_percentage, "%")

Cancelled Transactions : 8872
Percentage : 2.21 %


## Cleaning Decision 3

Cancelled transactions do not represent actual purchases.

Since recommendation systems should learn from successful purchases, cancellation transactions will be removed from the modeling dataset.

In [9]:
# ==================================================
# REMOVE CANCELLATION TRANSACTIONS
# ==================================================

rows_before = len(df_clean)

df_clean = df_clean[
    ~df_clean["InvoiceNo"]
    .astype(str)
    .str.startswith("C")
]

rows_after = len(df_clean)

print("Cancelled Rows Removed :", rows_before - rows_after)
print("Remaining Rows :", rows_after)

Cancelled Rows Removed : 8872
Remaining Rows : 392732


In [10]:
# ==================================================
# NEGATIVE QUANTITY CHECK
# ==================================================

negative_quantity = (
    df_clean["Quantity"] < 0
).sum()

negative_percentage = round(
    negative_quantity / len(df_clean) * 100,
    2
)

print("Negative Quantity Rows :", negative_quantity)
print("Percentage :", negative_percentage, "%")

Negative Quantity Rows : 0
Percentage : 0.0 %


## Cleaning Decision 4

Negative quantities represent product returns, refunds, or correction transactions.

Recommendation systems should learn only from positive purchase behavior; therefore, negative quantities will be removed.

In [11]:
# ==================================================
# REMOVE NEGATIVE QUANTITY
# ==================================================

rows_before = len(df_clean)

df_clean = df_clean[
    df_clean["Quantity"] > 0
]

rows_after = len(df_clean)

print("Negative Quantity Removed :", rows_before - rows_after)
print("Remaining Rows :", rows_after)

Negative Quantity Removed : 0
Remaining Rows : 392732


In [12]:
# ==================================================
# NEGATIVE PRICE CHECK
# ==================================================

negative_price = (
    df_clean["UnitPrice"] < 0
).sum()

negative_percentage = round(
    negative_price / len(df_clean) * 100,
    4
)

print("Negative Price Rows :", negative_price)
print("Percentage :", negative_percentage, "%")

Negative Price Rows : 0
Percentage : 0.0 %


In [14]:
# ==================================================
# ZERO PRICE CHECK
# ==================================================

zero_price = (df_clean["UnitPrice"] == 0).sum()

zero_price_percentage = round(
    zero_price / len(df_clean) * 100,
    4
)

print("Zero Price Rows :", zero_price)
print("Percentage :", zero_price_percentage, "%")

Zero Price Rows : 0
Percentage : 0.0 %


In [15]:
# ==================================================
# INSPECT ZERO PRICE TRANSACTIONS
# ==================================================

display(
    df_clean[df_clean["UnitPrice"] == 0].head(10)
)

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country


## Cleaning Decision 5

Transactions with negative prices do not represent valid sales.

These records will be removed to ensure that the recommendation system learns only from legitimate customer purchases.

In [13]:
# ==================================================
# REMOVE NEGATIVE PRICE
# ==================================================

rows_before = len(df_clean)

df_clean = df_clean[
    df_clean["UnitPrice"] > 0
]

rows_after = len(df_clean)

print("Negative Price Removed :", rows_before - rows_after)
print("Remaining Rows :", rows_after)

Negative Price Removed : 40
Remaining Rows : 392692


In [16]:
# ==================================================
# FINAL UNIT PRICE VALIDATION
# ==================================================

print("Minimum UnitPrice :", df_clean["UnitPrice"].min())
print("Rows with UnitPrice <= 0 :",
      (df_clean["UnitPrice"] <= 0).sum())

display(
    df_clean[df_clean["UnitPrice"] <= 0]
)

Minimum UnitPrice : 0.001
Rows with UnitPrice <= 0 : 0


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country


In [17]:
# ==================================================
# FINAL DATASET VALIDATION
# ==================================================

print("="*50)
print("FINAL DATASET VALIDATION")
print("="*50)

print("Final Shape :", df_clean.shape)
print("Missing Values :", df_clean.isnull().sum().sum())
print("Duplicate Rows :", df_clean.duplicated().sum())
print("Negative Quantity :", (df_clean["Quantity"] < 0).sum())
print("Non-Positive Price :", (df_clean["UnitPrice"] <= 0).sum())
print("Cancelled Transactions :",
      df_clean["InvoiceNo"].astype(str).str.startswith("C").sum())

FINAL DATASET VALIDATION
Final Shape : (392692, 8)
Missing Values : 0
Duplicate Rows : 0
Negative Quantity : 0
Non-Positive Price : 0
Cancelled Transactions : 0


In [19]:
# ==================================================
# SAVE CLEAN DATASET
# ==================================================

OUTPUT_PATH = (
    PROJECT_ROOT /
    "data" /
    "processed" /
    "online_retail_clean.csv"
)

df_clean.to_csv(
    OUTPUT_PATH,
    index=False
)

print("Clean dataset saved successfully.")
print("Location:", OUTPUT_PATH)

Clean dataset saved successfully.
Location: /home/egion/retail_product_recomendation/data/processed/online_retail_clean.csv


# Findings

1. A total of 135,080 rows containing missing CustomerID or Description were removed.

2. A total of 5,225 exact duplicate rows were identified and removed.

3. A total of 8,872 cancelled transactions were removed.

4. After removing cancelled transactions, no negative quantity records remained.

5. The minimum product price after cleaning is 0.001, indicating that all remaining transactions represent valid purchases.

6. The final cleaned dataset contains 392,692 valid customer-product interactions.

7. The dataset no longer contains missing values, duplicate records, invalid quantities, or cancellation transactions.

8. The cleaned dataset is suitable for recommendation system development.

# Decision

1. Records without customer identifiers were removed because customer-product interactions cannot be established.

2. Duplicate records were removed to avoid interaction bias.

3. Cancelled transactions were excluded because they do not represent successful purchases.

4. Only positive customer purchase interactions were retained.

5. Transactions with non-positive prices were excluded.

6. The cleaned dataset will serve as the master dataset for all subsequent analyses and model development.